In [ ]:
import pandas as pd

print("Pandas is working!")
print(pd.__version__)
import pandas as pd
import numpy as np

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

In [ ]:
import pandas as pd

DATA_PATH = "../data/raw/results.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Rows:", len(df))
print("Columns:", len(df.columns))

In [ ]:
import os

print(os.getcwd())


In [ ]:
import pandas as pd

df = pd.read_csv("../../data/raw/results.csv")

df.head()

In [ ]:
print(df.shape)

In [ ]:
import pandas as pd

df = pd.read_csv("../data/raw/results.csv")

print("Data loaded successfully!")

In [ ]:
df.info()


In [ ]:
df.columns.tolist()

In [ ]:
import pandas as pd

df = pd.read_csv("../data/raw/results.csv")

print("Data loaded successfully!")
print(df.head())

In [ ]:
import pandas as pd

df = pd.read_csv("../data/raw/results.csv")

print("Loaded successfully!")
print(df.columns.tolist())

In [ ]:
df.isnull().sum()

In [ ]:
def get_result(row):
    if row["home_score"] > row["away_score"]:
        return "H"
    elif row["home_score"] < row["away_score"]:
        return "A"
    else:
        return "D"

df["result"] = df.apply(get_result, axis=1)

In [ ]:
df[["home_team", "away_team", "home_score", "away_score", "result"]].head(10)

In [ ]:
df["result"].value_counts()

In [ ]:
df["date"] = pd.to_datetime(df["date"])

In [ ]:
df = df.sort_values("date").reset_index(drop=True)

In [ ]:
df[["date", "home_team", "away_team", "home_score", "away_score"]].head(10)

In [ ]:
df["home_points"] = 0
df["away_points"] = 0

df.loc[df["result"] == "H", "home_points"] = 3
df.loc[df["result"] == "A", "away_points"] = 3

df.loc[df["result"] == "D", "home_points"] = 1
df.loc[df["result"] == "D", "away_points"] = 1

print("Points calculated successfully!")

In [ ]:
df[[
    "home_team",
    "away_team",
    "result",
    "home_points",
    "away_points"
]].head(10)

In [ ]:
team_history = {}

home_form = []
away_form = []

for index, row in df.iterrows():

    home_team = row["home_team"]
    away_team = row["away_team"]

    # Get previous matches for each team
    home_history = team_history.get(home_team, [])
    away_history = team_history.get(away_team, [])

    # Calculate previous 5-match points
    home_recent_form = sum(home_history[-5:])
    away_recent_form = sum(away_history[-5:])

    home_form.append(home_recent_form)
    away_form.append(away_recent_form)

    # Add CURRENT match result to history AFTER calculating form
    team_history.setdefault(home_team, []).append(row["home_points"])
    team_history.setdefault(away_team, []).append(row["away_points"])

df["home_recent_form"] = home_form
df["away_recent_form"] = away_form

print("Recent form created!")

In [ ]:
home_recent_form = sum(home_history[-5:])

In [ ]:
team_history.setdefault(home_team, []).append(row["home_points"])

In [ ]:
df[
    [
        "date",
        "home_team",
        "away_team",
        "home_recent_form",
        "away_recent_form",
        "result"
    ]
].head(20)

In [ ]:
df["form_difference"] = (
    df["home_recent_form"] - df["away_recent_form"]
)

In [ ]:
df[
    [
        "home_team",
        "away_team",
        "home_recent_form",
        "away_recent_form",
        "form_difference",
        "result"
    ]
].head(20)

In [ ]:
team_stats = {}

home_strength = []
away_strength = []

for index, row in df.iterrows():

    home_team = row["home_team"]
    away_team = row["away_team"]

    # Get statistics BEFORE the current match
    home_stats = team_stats.get(
        home_team,
        {"points": 0, "matches": 0}
    )

    away_stats = team_stats.get(
        away_team,
        {"points": 0, "matches": 0}
    )

    # Calculate PPG before today's match
    if home_stats["matches"] > 0:
        home_ppg = home_stats["points"] / home_stats["matches"]
    else:
        home_ppg = 0

    if away_stats["matches"] > 0:
        away_ppg = away_stats["points"] / away_stats["matches"]
    else:
        away_ppg = 0

    home_strength.append(home_ppg)
    away_strength.append(away_ppg)

    # Update statistics AFTER calculating today's features
    team_stats.setdefault(
        home_team,
        {"points": 0, "matches": 0}
    )

    team_stats.setdefault(
        away_team,
        {"points": 0, "matches": 0}
    )

    team_stats[home_team]["points"] += row["home_points"]
    team_stats[home_team]["matches"] += 1

    team_stats[away_team]["points"] += row["away_points"]
    team_stats[away_team]["matches"] += 1

df["home_strength"] = home_strength
df["away_strength"] = away_strength

print("Team strength created!")

In [ ]:
df["strength_difference"] = (
    df["home_strength"] - df["away_strength"]
)

In [ ]:
df[
    [
        "date",
        "home_team",
        "away_team",
        "home_recent_form",
        "away_recent_form",
        "home_strength",
        "away_strength",
        "form_difference",
        "strength_difference",
        "result"
    ]
].head(20)

In [ ]:
df["neutral"] = df["neutral"].astype(int)

In [ ]:
features = [
    "home_recent_form",
    "away_recent_form",
    "form_difference",
    "home_strength",
    "away_strength",
    "strength_difference",
    "neutral"
]

print("Features created successfully!")

In [ ]:
df[features].head()

In [ ]:
X = df[features]
y = df["result"]

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
split_index = int(len(df) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print("Training matches:", len(X_train))
print("Testing matches:", len(X_test))

In [ ]:
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

In [ ]:
print(y_train.value_counts(normalize=True))

In [30]:
split_index = int(len(df) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 39616
Testing samples: 9904


In [31]:
print("Training outcomes:")
print(y_train.value_counts(normalize=True))

print("\nTesting outcomes:")
print(y_test.value_counts(normalize=True))

Training outcomes:
result
H    0.493462
A    0.280619
D    0.225919
Name: proportion, dtype: float64

Testing outcomes:
result
H    0.476070
A    0.290691
D    0.233239
Name: proportion, dtype: float64


In [32]:
from sklearn.linear_model import LogisticRegression

In [33]:
model = LogisticRegression(
    max_iter=1000
)

In [34]:
model.fit(X_train, y_train)

print("Model trained successfully!")

Model trained successfully!


In [35]:
y_pred = model.predict(X_test)

print(y_pred[:20])

['H' 'H' 'H' 'H' 'A' 'H' 'H' 'H' 'H' 'H' 'H' 'H' 'H' 'A' 'H' 'H' 'H' 'A'
 'A' 'H']


In [36]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)
print("Accuracy percentage:", accuracy * 100, "%")

Accuracy: 0.5581583198707593
Accuracy percentage: 55.815831987075924 %


In [37]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_test,
    y_pred,
    labels=["H", "D", "A"]
)

print(cm)

[[4245    1  469]
 [1792    0  518]
 [1596    0 1283]]


In [38]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        y_pred,
        labels=["H", "D", "A"]
    )
)

              precision    recall  f1-score   support

           H       0.56      0.90      0.69      4715
           D       0.00      0.00      0.00      2310
           A       0.57      0.45      0.50      2879

    accuracy                           0.56      9904
   macro avg       0.37      0.45      0.40      9904
weighted avg       0.43      0.56      0.47      9904

